## 1. Загрузка и первичная проверка данных

Для анализа A/B-теста используем:

- подготовленную ранее витрину страховых договоров;
- таблицу участников эксперимента `experiment_group`.

Предыдущая витрина уже содержит очищенные данные по договорам и страховым выплатам, поэтому повторно выполнять их подготовку не требуется.

Таблица `experiment_group` будет основной для определения состава контрольной и тестовой групп и последующего расчёта конверсии.

In [1]:
import pandas as pd
import numpy as np

insurance = pd.read_excel("cleaned_insurance_data.xlsx")
experiment = pd.read_excel("experiment_group.xlsx")

print("Размер страховой витрины:", insurance.shape)
print("Размер таблицы эксперимента:", experiment.shape)

print("\nСтолбцы страховой витрины:")
print(insurance.columns.tolist())

print("\nСтолбцы experiment_group:")
print(experiment.columns.tolist())

Размер страховой витрины: (3711, 17)
Размер таблицы эксперимента: (14842, 3)

Столбцы страховой витрины:
['contract_num', 'product_name', 'client_id', 'contract_status', 'currency_name', 'duration', 'country', 'age', 'sex', 'price', 'insurance_amount', 'loss_payout_amt', 'price_usd', 'insurance_amount_usd', 'loss_payout_amt_usd', 'premium_rate_per_day', 'anomaly']

Столбцы experiment_group:
['client_id', 'group', 'experiment_id']


## 2. Проверка состава эксперимента

Проверим:

- какие эксперименты присутствуют в таблице;
- какие значения содержит признак `group`;
- количество участников в контрольной и тестовой группах;
- наличие пропусков и дубликатов по клиентам.

In [2]:
print("Уникальные experiment_id:")
print(experiment["experiment_id"].value_counts(dropna=False))

print("\nРаспределение групп:")
print(experiment["group"].value_counts(dropna=False))

print("\nКоличество пропусков:")
print(experiment.isna().sum())

print(
    "\nКоличество дубликатов client_id:",
    experiment["client_id"].duplicated().sum()
)

display(experiment.head())

Уникальные experiment_id:
experiment_id
policy_pricing    14842
Name: count, dtype: int64

Распределение групп:
group
test       6306
control    6043
NaN        2493
Name: count, dtype: int64

Количество пропусков:
client_id           0
group            2493
experiment_id       0
dtype: int64

Количество дубликатов client_id: 0


,client_id,group,experiment_id
0,10001718,control,policy_pricing
1,1017180395,control,policy_pricing
2,1017178047,control,policy_pricing
3,1017174614,control,policy_pricing
4,1017174071,control,policy_pricing


## 3. Подготовка участников A/B-теста

В таблице присутствует один эксперимент - `policy_pricing`.

Для анализа оставим только клиентов, распределённых в контрольную (`control`) или тестовую (`test`) группу.

Строки с отсутствующим значением `group` исключаем из A/B-сравнения, поскольку невозможно определить вариант эксперимента, который был показан таким клиентам.

Перед объединением со страховой витриной проверим размер групп и пересечение клиентов между таблицами.

In [3]:
ab_users = experiment[
    experiment["group"].isin(["control", "test"])
].copy()

print("Количество участников A/B-теста:", len(ab_users))

print("\nРазмер групп:")
print(ab_users["group"].value_counts())

print(
    "\nУникальных участников:",
    ab_users["client_id"].nunique()
)

print(
    "\nКлиентов из страховой витрины, найденных в A/B-тесте:",
    insurance["client_id"].isin(ab_users["client_id"]).sum()
)

Количество участников A/B-теста: 12349

Размер групп:
group
test       6306
control    6043
Name: count, dtype: int64

Уникальных участников: 12349

Клиентов из страховой витрины, найденных в A/B-тесте: 3616


## 4. Сбор A/B-витрины

Для расчёта конверсии необходимо сохранить всех участников эксперимента, включая тех, кто не оформил страховой договор.

Поэтому таблицу участников A/B-теста используем как основную и присоединяем к ней страховую витрину через `left join` по `client_id`.

Если после объединения у клиента отсутствует номер договора, считаем, что договор не был оформлен.

In [4]:
ab_data = ab_users.merge(
    insurance,
    on="client_id",
    how="left"
)

ab_data["converted"] = (
    ab_data["contract_num"].notna()
).astype(int)

print("Размер A/B-витрины:", ab_data.shape)

print("\nРаспределение converted:")
print(ab_data["converted"].value_counts())

print("\nКонверсия в целом:")
print(round(ab_data["converted"].mean() * 100, 2), "%")

display(
    ab_data[
        [
            "client_id",
            "group",
            "contract_num",
            "converted"
        ]
    ].head(10)
)

Размер A/B-витрины: (12349, 20)

Распределение converted:
converted
0    8733
1    3616
Name: count, dtype: int64

Конверсия в целом:
29.28 %


,client_id,group,contract_num,converted
0,10001718,control,ТТЕ7227715*****,1
1,1017180395,control,ДСЖ-5/1539*****,1
2,1017178047,control,ТМБЕ579 № *****,1
3,1017174614,control,ТМГ1535Е7 *****,1
4,1017174071,control,ТТЕ7227777*****,1
5,1017173064,control,ДСЖ-5/1539*****,1
6,10137843967,control,ИТЕ7 67777*****,1
7,1017171040,control,ДСЖ-5/1539*****,1
8,10137878976,control,ДСЖ-5/1539*****,1
9,10137937581,control,ДСЖ-5/1539*****,1


## 5. Сравнение конверсии между группами

Сравним конверсию в оформление страхового договора между контрольной и тестовой группами.

Конверсию рассчитываем как долю участников эксперимента, у которых был оформлен договор:

`conversion = количество оформивших договор / количество участников группы`

In [5]:
conversion_by_group = (
    ab_data.groupby("group")["converted"]
    .agg(["count", "sum", "mean"])
)

conversion_by_group["conversion_pct"] = (
    conversion_by_group["mean"] * 100
)

display(conversion_by_group.round(4))

,count,sum,mean,conversion_pct
group,,,,
control,6043,1949,0.3225,32.2522
test,6306,1667,0.2644,26.4351


In [6]:
control_conversion = (
    ab_data.loc[
        ab_data["group"] == "control",
        "converted"
    ].mean()
)

test_conversion = (
    ab_data.loc[
        ab_data["group"] == "test",
        "converted"
    ].mean()
)

absolute_diff = test_conversion - control_conversion

relative_diff = (
    absolute_diff / control_conversion
)

print(
    "Конверсия control:",
    round(control_conversion * 100, 2),
    "%"
)

print(
    "Конверсия test:",
    round(test_conversion * 100, 2),
    "%"
)

print(
    "Абсолютная разница:",
    round(absolute_diff * 100, 2),
    "п.п."
)

print(
    "Относительное изменение:",
    round(relative_diff * 100, 2),
    "%"
)

Конверсия control: 32.25 %
Конверсия test: 26.44 %
Абсолютная разница: -5.82 п.п.
Относительное изменение: -18.04 %


### Вывод

Конверсия в оформление договора в контрольной группе составила 32.25%, а в тестовой - 26.44%.

Абсолютное изменение составило -5.82 п.п., относительное - около -18.04%.

Таким образом, в тестовой группе наблюдается снижение конверсии. Далее проверим, является ли это различие статистически значимым.

## 6. Статистическая проверка различий в конверсии

Проверим, является ли наблюдаемая разница между контрольной и тестовой группами статистически значимой.

Сформулируем гипотезы:

- **H0:** средняя конверсия в контрольной и тестовой группах одинакова;
- **H1:** средняя конверсия между группами различается.

Уровень значимости принимаем равным `α = 0.05`.

Так как признак `converted` является бинарным (`0/1`), его среднее значение соответствует конверсии.

In [7]:
from scipy.stats import ttest_ind

control = ab_data.loc[
    ab_data["group"] == "control",
    "converted"
]

test = ab_data.loc[
    ab_data["group"] =="test",
    "converted"
]

t_stat, p_value = ttest_ind(
    control,
    test,
    equal_var=False
)

print("t-statistic:", round(t_stat, 4))
print("p-value:", round(p_value, 6))

alpha = 0.05

if p_value < alpha:
    print("Различие статистически значимо.")
else:
    print("Статистически значимого различия не обнаружено.")

t-statistic: 7.1063
p-value: 0.0
Различие статистически значимо.


### Вывод

Полученное `p-value < 0.001`, поэтому нулевая гипотеза об отсутствии различий между группами отвергается.

Снижение конверсии с 32.25% в контрольной группе до 26.44% в тестовой группе является статистически значимым.

Таким образом, новое ценообразование связано со снижением конверсии примерно на 5.82 п.п. Однако для итогового бизнес-решения необходимо дополнительно оценить стоимость полиса и убыточность.

## 7. Сравнение стоимости полиса

Следующей ключевой метрикой является стоимость оформленного страхового полиса.

Поскольку данные о фактической стоимости договора присутствуют только для клиентов, оформивших полис, сравнение выполняем среди записей с `converted = 1`.

Для сопоставимости используем стоимость полиса в долларах США - `price_usd`.

In [8]:
converted_data = ab_data[
    ab_data["converted"] == 1
].copy()

price_by_group = (
    converted_data.groupby("group")["price_usd"]
    .agg(["count", "mean", "median", "std"])
)

display(price_by_group.round(2))

,count,mean,median,std
group,,,,
control,1949,47.70,22.35,134.82
test,1667,59.93,17.38,158.71


In [9]:
control_price = (
    converted_data.loc[
        converted_data["group"] == "control",
        "price_usd"
    ].mean()
)

test_price = (
    converted_data.loc[
        converted_data["group"] == "test",
        "price_usd"
    ].mean()
)

price_diff = test_price - control_price

price_relative_diff = (
    price_diff / control_price
)

print("Средняя цена control:", round(control_price, 2), "USD")
print("Средняя цена test:", round(test_price, 2), "USD")
print("Абсолютная разница:", round(price_diff, 2), "USD")
print(
    "Относительное изменение:",
    round(price_relative_diff * 100, 2),
    "%"
)

Средняя цена control: 47.7 USD
Средняя цена test: 59.93 USD
Абсолютная разница: 12.23 USD
Относительное изменение: 25.64 %


### Вывод

Средняя стоимость оформленного полиса в контрольной группе составляет 47.70 USD, а в тестовой - 59.93 USD.

Таким образом, в тестовой группе средняя стоимость полиса выше на 12.23 USD, или примерно на 25.64%.

При этом ранее было выявлено статистически значимое снижение конверсии в тестовой группе. Поэтому для итоговой оценки нового подхода необходимо дополнительно проверить, компенсирует ли рост стоимости полиса снижение количества оформлений и как изменяется убыточность.

## 8. Оценка выручки на одного участника эксперимента

Для бизнес-оценки нового ценообразования недостаточно рассматривать только среднюю стоимость оформленного полиса.

В тестовой группе цена полиса выше, но конверсия ниже. Поэтому рассчитаем ожидаемую выручку на одного участника эксперимента:

`revenue_per_user = conversion × average_price`

Такой показатель учитывает одновременно вероятность оформления договора и среднюю стоимость полиса.

In [10]:
control_revenue_per_user = (
    control_conversion * control_price
)

test_revenue_per_user = (
    test_conversion * test_price
)

revenue_diff = (
    test_revenue_per_user
    - control_revenue_per_user
)

revenue_relative_diff = (
    revenue_diff
    / control_revenue_per_user
)

print(
    "Выручка на пользователя control:",
    round(control_revenue_per_user, 2),
    "USD"
)

print(
    "Выручка на пользователя test:",
    round(test_revenue_per_user, 2),
    "USD"
)

print(
    "Абсолютная разница:",
    round(revenue_diff, 2),
    "USD"
)

print(
    "Относительное изменение:",
    round(revenue_relative_diff * 100, 2),
    "%"
)

Выручка на пользователя control: 15.38 USD
Выручка на пользователя test: 15.84 USD
Абсолютная разница: 0.46 USD
Относительное изменение: 2.98 %


### Вывод

Несмотря на снижение конверсии в тестовой группе, увеличение средней стоимости полиса компенсирует этот эффект.

Ожидаемая выручка на одного участника эксперимента составляет 15.38 USD в контрольной группе и 15.84 USD в тестовой.

Таким образом, в тестовой группе показатель выше примерно на 0.46 USD, или на 2.98%.

Однако для итоговой оценки нового ценообразования необходимо учитывать страховые выплаты и сравнить убыточность групп.

## 9. Сравнение убыточности

Для оценки финансового результата сравним убыточность контрольной и тестовой групп.

Убыточность рассчитываем как отношение общей суммы страховых выплат к общей сумме полученных страховых премий:

`loss_ratio = total_loss / total_premium`

Расчёт выполняем только для оформленных договоров.

In [11]:
loss_by_group = (
    converted_data.groupby("group")
    .agg(
        contracts=("contract_num", "count"),
        total_premium=("price_usd", "sum"),
        total_loss=("loss_payout_amt_usd", "sum")
    )
)

loss_by_group["loss_ratio"] = (
    loss_by_group["total_loss"]
    / loss_by_group["total_premium"]
)

loss_by_group["loss_ratio_pct"] = (
    loss_by_group["loss_ratio"] * 100
)

display(loss_by_group.round(2))

,contracts,total_premium,total_loss,loss_ratio,loss_ratio_pct
group,,,,,
control,1949,92968.33,72285.4,0.78,77.75
test,1667,99908.03,88408.7,0.88,88.49


In [12]:
control_loss_ratio = loss_by_group.loc[
    "control", "loss_ratio"
]

test_loss_ratio = loss_by_group.loc[
    "test", "loss_ratio"
]

loss_ratio_diff = (
    test_loss_ratio - control_loss_ratio
)

print(
    "Убыточность control:",
    round(control_loss_ratio * 100, 2),
    "%"
)

print(
    "Убыточность test:",
    round(test_loss_ratio * 100, 2),
    "%"
)

print(
    "Разница:",
    round(loss_ratio_diff * 100, 2),
    "п.п."
)

Убыточность control: 77.75 %
Убыточность test: 88.49 %
Разница: 10.74 п.п.


### Вывод

Убыточность контрольной группы составляет 77.75%, а тестовой - 88.49%.

Таким образом, в тестовой группе доля страховых выплат в собранной премии выше на 10.74 п.п.

Несмотря на более высокую стоимость полиса и небольшое увеличение выручки на одного участника эксперимента, рост убыточности может ухудшать финансовый результат нового подхода.

Для итоговой оценки рассчитаем финансовый результат на одного участника эксперимента с учётом страховых выплат.

## 10. Финансовый результат на одного участника эксперимента

Для итоговой бизнес-оценки рассчитаем финансовый результат как разницу между полученной страховой премией и страховыми выплатами.

Для клиентов, которые не оформили договор, премия и выплаты принимаются равными нулю.

`financial_result = premium - loss`

Затем сравним средний финансовый результат на одного участника контрольной и тестовой групп.

In [13]:
ab_data["premium"] = ab_data["price_usd"].fillna(0)
ab_data["loss"] = ab_data["loss_payout_amt_usd"].fillna(0)

ab_data["financial_result"] = (
    ab_data["premium"] - ab_data["loss"]
)

financial_by_group = (
    ab_data.groupby("group")["financial_result"]
    .agg(["count", "sum", "mean"])
)

display(financial_by_group.round(2))

,count,sum,mean
group,,,
control,6043,20682.93,3.42
test,6306,11499.33,1.82


In [14]:
control_financial = financial_by_group.loc[
    "control", "mean"
]

test_financial = financial_by_group.loc[
    "test", "mean"
]

financial_diff = (
    test_financial - control_financial
)

financial_relative_diff = (
    financial_diff / control_financial
)

print(
    "Финансовый результат на участника control:",
    round(control_financial, 2),
    "USD"
)

print(
    "Финансовый результат на участника test:",
    round(test_financial, 2),
    "USD"
)

print(
    "Абсолютная разница:",
    round(financial_diff, 2),
    "USD"
)

print(
    "Относительное изменение:",
    round(financial_relative_diff * 100, 2),
    "%"
)

Финансовый результат на участника control: 3.42 USD
Финансовый результат на участника test: 1.82 USD
Абсолютная разница: -1.6 USD
Относительное изменение: -46.72 %


### Вывод

Средний финансовый результат на одного участника эксперимента составил 3.42 USD в контрольной группе и 1.82 USD в тестовой.

Таким образом, тестовая группа показывает снижение финансового результата примерно на 1.60 USD на участника, или на 46.72%.

Рост средней стоимости полиса не компенсирует одновременно снижение конверсии и более высокую убыточность.

## 11. Статистическая проверка финансового результата

Проверим, является ли различие среднего финансового результата на одного участника между контрольной и тестовой группами статистически значимым.

- **H0:** средний финансовый результат в группах одинаков;
- **H1:** средний финансовый результат различается.

Уровень значимости: `α = 0.05`.

In [15]:
control_financial_values = ab_data.loc[
    ab_data["group"] == "control",
    "financial_result"
]

test_financial_values = ab_data.loc[
    ab_data["group"] == "test",
    "financial_result"
]

t_stat_fin, p_value_fin = ttest_ind(
    control_financial_values,
    test_financial_values,
    equal_var=False
)

print("t-statistic:", round(t_stat_fin, 4))
print("p-value:", round(p_value_fin, 6))

if p_value_fin < 0.05:
    print("Различие статистически значимо.")
else:
    print("Статистически значимого различия не обнаружено.")

t-statistic: 0.2056
p-value: 0.837136
Статистически значимого различия не обнаружено.


### Вывод

Средний финансовый результат на одного участника в контрольной группе выше, чем в тестовой: 3.42 USD против 1.82 USD.

Однако `p-value = 0.8371`, что значительно выше уровня значимости `0.05`.

Следовательно, статистически значимого различия финансового результата между группами не обнаружено.

Таким образом, несмотря на наблюдаемое снижение финансового результата в тестовой группе, имеющихся данных недостаточно, чтобы считать это различие статистически подтверждённым.

## 12. Итоговый вывод и рекомендация

В ходе A/B-теста были сравнены контрольная и тестовая схемы ценообразования страхового продукта.

Основные результаты:

- конверсия в оформление договора снизилась с 32.25% в контрольной группе до 26.44% в тестовой;
- снижение конверсии составило 5.82 п.п. и является статистически значимым;
- средняя стоимость оформленного полиса выросла с 47.70 USD до 59.93 USD, то есть примерно на 25.64%;
- выручка на одного участника эксперимента выросла незначительно - примерно на 2.98%;
- убыточность увеличилась с 77.75% до 88.49%;
- средний финансовый результат на одного участника снизился с 3.42 USD до 1.82 USD.

При этом статистически значимого различия по финансовому результату на одного участника не обнаружено: `p-value = 0.8371`.

Таким образом, тестовая схема приводит к статистически значимому снижению конверсии и одновременно показывает более высокую убыточность. Рост средней стоимости полиса не даёт убедительного улучшения итогового финансового результата.

По результатам текущего эксперимента оснований для перехода на новую схему ценообразования недостаточно. Целесообразно сохранить текущий подход и дополнительно доработать новую модель ценообразования перед повторным тестированием.